### Step 1: Data Ingestion and Pre-processing
<p>This section contains the data loader with support for doc, docx and pdf file formats.<br>
<b>Step 1: Data Loading and Serialization</b><br>
All files contained within the INPUT_FOLDER specified at the top are loaded, parsed, serialized to JSON and saved to the <u>parsed</u> folder.<br>
<b>Step 2: Data Cleaning</b><br>
All JSON files within the parsed folders are cleaned in-place to remove unncessary whitespaces and elements with empty text.<br>
<b>Step 3: Data Normalization</b><br>
All cleaned JSON files within the parsed folders are now normalized to have a uniform 4 key schema, which includes type, text, coursename and coursecode and stored in the <u>normalized</u> folder.
</p>

In [ ]:
import os

# --- import your functions ---
from src.ingestion_and_preprocessing import *

# --- folder containing raw documents ---
INPUT_FOLDER = "dataset"

# --- process all files ---
for file_name in os.listdir(INPUT_FOLDER):
    file_path = os.path.join(INPUT_FOLDER, file_name)

    # skip non-files
    if not os.path.isfile(file_path):
        continue

    # --- only supported formats ---
    if not file_name.lower().endswith((".pdf", ".doc", ".docx")):
        continue

    print(f"\nProcessing: {file_name}")

    try:
        # 1. Partition → saves to parsed/
        partitioner(file_path)

        # --- construct parsed file path ---
        name, ext = os.path.splitext(file_name)
        ext = ext.lower()

        if ext == ".pdf":
            parsed_path = f"parsed/{name}_pdf.json"
        elif ext == ".doc":
            parsed_path = f"parsed/{name}_doc.json"
        elif ext == ".docx":
            parsed_path = f"parsed/{name}_docx.json"

        # 2a. Clean (in-place)
        clean_json_elements(parsed_path)

        # 2b. Table → Markdown (in-place)
        convert_tables_to_markdown(parsed_path)

        # 3. Normalize → saves to normalized/
        normalize_json_elements(parsed_path)

    except Exception as e:
        print(f"Error processing {file_name}: {e}")

### Step 2: Chunking
<p> Fixed token chunking strategy was employed with 2 chunk sizes and overlaps: <b>256/26</b> and <b>512/52</b>.<br>
Chunking was done by employing two types of tokenizers as per those used by our embedding models, namely <i>sentence-transformers/all-MiniLM-L6-v2</i> and <i>BAAI/bge-base-en-v1.5</i>.<br>
Three types of chunks were made: 256/26 chunk for both MiniLM-L6-v2 and bge-base-en-v1.5 and then 512/52 for bge-base-en-v1.5 alone.
</p>

In [ ]:
from src.chunking import chunking_minilm_l6_v2

chunking_minilm_l6_v2(
    input_folder="normalized",
    chunk_size=256,
    overlap=26
)

In [ ]:
from src.chunking import chunking_bge_base_v1_5

chunking_bge_base_v1_5(
    input_folder="normalized",
    chunk_size=256,
    overlap=26
)

In [ ]:
chunking_bge_base_v1_5(
    input_folder="normalized",
    chunk_size=512,
    overlap=52
)

### Step 3: Embedding and Vector Database Storage

In [ ]:
from src.embedding_qdrant import embed_and_store

embed_and_store(
    chunk_folder="chunks/minilm_l6_v2_256_26",
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    collection_name="minilm_l6_v2_256_26",
)

embed_and_store(
    chunk_folder="chunks/bge_base_v1_5_256_26",
    model_name="BAAI/bge-base-en-v1.5",
    collection_name="bge_base_v1_5_256_26",
)

embed_and_store(
    chunk_folder="chunks/bge_base_v1_5_512_52",
    model_name="BAAI/bge-base-en-v1.5",
    collection_name="bge_base_v1_5_512_52",
)

In [ ]:
from src.embedding_chromadb import embed_and_store

embed_and_store(
    chunk_folder="chunks/bge_base_v1_5_512_52",
    model_name="BAAI/bge-base-en-v1.5",
    collection_name="bge_base_v1_5_512_52",
)

### Step 4: LLM Prompting and Response Generation

In [ ]:
from src.rag_prompt import rag_inference
from src.query import *

query = "What is the assessment pattern in detail with all components of generative ai?"

chunks = query_qdrant(
    query=query,
    model_name="BAAI/bge-base-en-v1.5",
    collection_name="bge_base_v1_5_512_52",
)

answer = rag_inference(
    model_name="llama3.1:8b",
    retrieved_chunks=chunks,
    query=query
)

print(answer)

In [ ]:
from src.rag_prompt import rag_inference
from src.query import *

query = "What is the assessment pattern in detail with all components of generative ai?"

chunks = query_chromadb(
    query=query,
    model_name="BAAI/bge-base-en-v1.5",
    collection_name="bge_base_v1_5_512_52",
)

answer = rag_inference(
    model_name="llama3.1:8b",
    retrieved_chunks=chunks,
    query=query
)

print(answer)

In [ ]:
from src.rag_prompt import rag_inference
from src.query import *

query = "What is the assessment pattern in detail with all components of generative ai?"

chunks = query_chromadb(
    query=query,
    model_name="BAAI/bge-base-en-v1.5",
    collection_name="bge_base_v1_5_512_52",
)

answer = rag_inference(
    model_name="gemma2:9b",
    retrieved_chunks=chunks,
    query=query
)

print(answer)

In [ ]:
from src.rag_prompt import rag_inference
from src.query import *

query = "What is the assessment pattern in detail with all components of generative ai?"

chunks = query_chromadb(
    query=query,
    model_name="BAAI/bge-base-en-v1.5",
    collection_name="bge_base_v1_5_512_52",
)

answer = rag_inference(
    model_name="phi3:mini",
    retrieved_chunks=chunks,
    query=query
)

print(answer)

### Retrieval Quality Evaluation

In [2]:
from src.retrieval_evaluation import *

generate_eval_dataset(
    normalized_folder="normalized",
    num_files=15,
    samples_per_file=3,
    model_name="phi3:mini",
    save_path="eval_dataset.json"
)

[1/15] Processing file: Computer Organization & Architecture_CSE2709.json...
⚠️ Skipped malformed output: 
```json

{

    "question": "What levels of correlation are mentioned in the text?",

    "answer": ["low correlation", "medium correlation", "high correlation"]

}

```
⚠️ Skipped malformed output: 
**Question:** What is the mode of quizzes for End Term Exam?  

**Answer:** Offline.
[2/15] Processing file: Logical Reasoning & Quantitative Analysis_PSP4706.json...
[3/15] Processing file: Project-III_PRJ3902.json...
[4/15] Processing file: Making Causal Inferences to Support Decisions_CSE3025.json...
[5/15] Processing file: Network Security_CSE3714.json...
⚠️ Skipped malformed output: 
**Question:** In which year was the seventh edition of "Cryptography and Network Security" by William Stallings published?

**Answer:** The seventh edition was published in 2de/Jan - Mar 2017.
[6/15] Processing file: Design Thinking Course Handout_2025.json...
[7/15] Processing file: Generative AI an

[{'question': 'What types of computer components are mentioned as Input and Output devices?',
  'ground_truth_answer': 'The text mentions input devices without specifying which ones; however, it does state that output devices are also considered along with secondary storage devices. Without additional context from the original passage about specific device names or categories within each type (input/output), a precise answer cannot be provided solenerly based on this snippet of text alone.',
  'reference_text': 'Input devices, Output devices, Secondary storage devices.',
  'source_file': 'Computer Organization & Architecture_CSE2709'},
 {'question': 'What are some examples of Input and Output devices mentioned as well as a different category referenced in the text?',
  'ground_truth_answer': 'The input device is not specified; however, output devices such as monitors or speakers are implied. Secondary storage devices like hard drives are also referred to in the text.',
  'reference_tex

In [3]:
from src.retrieval_evaluation import *

generate_eval_dataset(
    normalized_folder="normalized",
    num_files=15,
    samples_per_file=3,
    model_name="mistral:7b-instruct ",
    save_path="eval_dataset_mistral.json"
)

[1/15] Processing file: Seminar Case Studies_SEM3501.json...
[2/15] Processing file: Making Causal Inferences to Support Decisions_CSE3025.json...
[3/15] Processing file: Financial Markets From Fundamentals to AI-Driven Analytics_FIN1001.json...
[4/15] Processing file: Cryptography_CSE3703.json...
[5/15] Processing file: Generative AI and LLMs_CSE3720.json...
[6/15] Processing file: Discrete Mathematics_CSE3715.json...
[7/15] Processing file: Computer Organization & Architecture_CSE2709.json...
[8/15] Processing file: Project-III_PRJ3902.json...
[9/15] Processing file: Practice School-II_PSI2501.json...
[10/15] Processing file: Innovation and Entrepreneurship_ENT3005.json...
[11/15] Processing file: Global Energy Politics, Markets and Policy_PSP2706.json...
[12/15] Processing file: Design Thinking Course Handout_2025.json...
[13/15] Processing file: Software Engineering_CSE3005.json...
[14/15] Processing file: Human Computer Interaction_CSE2724.json...
[15/15] Processing file: Network 

[{'question': "What is the time complexity of Linear Search, Maximum in an Array, Sorting (Selection, Bubble and Insertion), Binary search, Kadane's Algo, and Merge two sorted arrays?",
  'ground_truth_answer': "O(N) for Linear Search, Maximum in an Array, Sorting (Selection, Bubble and Insertion), and Merge two sorted arrays. Binary search and Kadane's Algo have a time complexity of O(N).",
  'reference_text': "• Linear Search, Maximum in an Array, Sorting (Selection, Bubble and Insertion), Binary search, Kadane's Algo- O(N), Merge two sorted arrays",
  'source_file': 'Seminar Case Studies_SEM3501'},
 {'question': 'What is the course number for the given text?',
  'ground_truth_answer': 'SEM 3501',
  'reference_text': 'Course Number: SEM 3501',
  'source_file': 'Seminar Case Studies_SEM3501'},
 {'question': 'What are the credits given for this item?',
  'ground_truth_answer': 'The credits given for this item are 2.',
  'reference_text': 'Credits: 2 (2-0-0)',
  'source_file': 'Seminar 

In [ ]:
from src.retrieval_evaluation import *

generate_eval_dataset(
    normalized_folder="normalized",
    num_files=15,
    samples_per_file=3,
    model_name="mistral:7b-instruct ",
    save_path="eval_dataset_mistral.json"
)